# AGC ttbar analysis with ROOT RDataFrame on Dask

This notebook runs the CMS ttbar analysis of the [Analysis Grand Challenge](https://agc.readthedocs.io) (AGC)
with **ROOT RDataFrame**, distributed over a **Dask** cluster. It is adapted from the ROOT team's AGC implementation
([root-project/analysis-grand-challenge](https://github.com/root-project/analysis-grand-challenge); see the references
at the end). The same notebook runs

- **on Spider or Snellius with RS-DAT** (the default), where the Dask workers are Slurm jobs and the input files are
  read directly from the AGC storage at CERN; and
- **locally**, on a Dask `LocalCluster` and a small skim of one input file, to develop and test.

Only the cluster and the input list change between the two; the analysis code is the same.

**What the analysis does.** Events with exactly one tight electron or muon and at least four jets are split into two
regions: exactly one b-tagged jet (`4j1b`, histogram of $H_T$) and at least two (`4j2b`, histogram of the mass of the
trijet with the highest $p_T$). For nominal samples, jet energy scale and resolution variations and b-tagging weight
variations are computed in the same pass over the data, with RDataFrame's `Vary`.

**How it is distributed.** RDataFrame builds a computation graph per sample; nothing is read until the graphs run.
With an `executor=client` argument, each graph is split into partitions (ranges of entries), every partition becomes
a Dask task, and the partial histograms are merged back in this notebook. The analysis code lives in `../agc/`.

## 1. Settings

The defaults run on the Slurm cluster (RS-DAT) and read the first AGC file of every sample from CERN EOS.
For local development, set `RUN_ON = "local"` and `DATA = "skim"`.

In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))  # the agc/ package next to notebooks/

RUN_ON = "rsdat"  # "rsdat": Dask workers as Slurm jobs (RS-DAT) | "local": Dask LocalCluster on this machine
N_WORKERS = 4      # local: single-threaded worker processes on this machine
N_JOBS = 2         # rsdat: Slurm jobs; each runs several workers (see scripts/jupyterdask-spider.slurm)

DATA = "remote"    # "remote": AGC files read over the network | "skim": one local file you downloaded
SKIM_PATH = Path("~/dev/data/hpc-dat/agc_ttbar_skim_100k.root").expanduser()  # used when DATA == "skim"
N_FILES_PER_SAMPLE = 1  # used when DATA == "remote": first N files of each of the 9 samples
PREFIX = "eos"     # "eos" (root://, CERN EOS public mirror) | "unl" (HTTPS, UNL) | URL prefix of your own copy

NPARTITIONS = None  # tasks per sample; None lets ROOT choose (one per worker core)
OUTPUT = Path("histograms.root")

## 2. Start a Dask cluster

**Local:** a `LocalCluster` with single-threaded worker processes (distributed RDataFrame runs one task per worker thread, and works best with one thread per process).

**RS-DAT:** the Jupyter server runs as a Slurm job started with `jupyterdask` and
`scripts/jupyterdask-spider.slurm`. That script sets the `DASK_JOBQUEUE__SLURM__*` defaults (cores, memory,
walltime, and the container image for the workers), so `SLURMCluster()` needs no arguments here.
You can also create the cluster from the Dask tab in JupyterLab's sidebar and connect to it with
`Client("tcp://...")` instead.

Note: `cores`, `processes` and `memory` are passed to `SLURMCluster(...)` explicitly (read from that Dask config),
because distributed RDataFrame takes the number of cores from the cluster's own arguments.

In [ ]:
from distributed import Client

if RUN_ON == "local":
    from distributed import LocalCluster
    cluster = LocalCluster(n_workers=N_WORKERS, threads_per_worker=1, processes=True)
    n_expected = N_WORKERS
elif RUN_ON == "rsdat":
    import dask
    from dask_jobqueue import SLURMCluster
    # Pass cores/processes/memory explicitly (values still come from the DASK_JOBQUEUE__SLURM__* settings in the
    # Slurm script): distributed RDataFrame reads the core count from the arguments the cluster was created with,
    # and fails with "Could not retrieve the provided worker specification" if they are only in the Dask config.
    slurm = dask.config.get("jobqueue.slurm")
    cluster = SLURMCluster(cores=slurm["cores"], processes=slurm["processes"], memory=slurm["memory"])
    cluster.scale(jobs=N_JOBS)
    n_expected = N_JOBS * (dask.config.get("jobqueue.slurm.processes") or 1)
else:
    raise ValueError(f"unknown RUN_ON: {RUN_ON}")

client = Client(cluster)
client.wait_for_workers(n_expected, timeout=900)  # Slurm jobs can take a while to start
print("Dashboard:", client.dashboard_link)
client

## 3. Choose the inputs

The AGC file list (`agc/data/nanoaod_inputs.json`) has 9 samples: ttbar (nominal and 4 systematic variations),
three single-top channels and W+jets. 691 files, ~3.5 TB in total; the first files of all 9 samples together are ~30 GB.
RDataFrame reads only the ~20 branches the analysis needs, so far less than that goes over the network.

A local skim is treated as one ttbar sample and normalised by its own number of events.

In [ ]:
from agc.inputs import local_inputs, remote_inputs

if DATA == "skim":
    inputs = local_inputs(SKIM_PATH)
elif DATA == "remote":
    inputs = remote_inputs(N_FILES_PER_SAMPLE, prefix=PREFIX)
else:
    raise ValueError(f"unknown DATA: {DATA}")

for inp in inputs:
    print(f"{inp.process:18s} {inp.variation:10s} {len(inp.paths):3d} file(s) {inp.nevents:>12,} events")

## 4. Run the analysis

`run_analysis` books the histograms for every sample, asks for the systematic variations, and then runs all
computation graphs together with `RunGraphs`. Watch the Dask dashboard while it runs.

In [ ]:
import time
import ROOT
from agc.rdf_analysis import run_analysis, save_histos

ROOT.gROOT.SetBatch(True)
ROOT.TH1.AddDirectory(False)

start = time.time()
histos = run_analysis(inputs, client=client, npartitions=NPARTITIONS)
elapsed = time.time() - start
nevents = sum(inp.nevents for inp in inputs)
print(f"{len(histos)} histograms from {nevents:,} events in {elapsed:.1f} s ({nevents / elapsed:,.0f} events/s)")

save_histos(histos, OUTPUT)
print("Saved to", OUTPUT.resolve())

## 5. Look at the results

In [ ]:
from agc.validation import plot_regions

fig = plot_regions(OUTPUT)

In [ ]:
for name in ["4j1b_ttbar", "4j2b_ttbar"]:
    h = histos[name]
    print(f"{name:12s} weighted events {h.Integral():10.1f}   raw entries {int(h.GetEntries()):7d}")

## 6. Validate against the AGC reference

The AGC publishes reference histograms for the first N files per sample. A run with `DATA = "remote"` and
`N_FILES_PER_SAMPLE` in (1, 2, 5, 10) can be compared directly. A skim cannot: it has fewer events and a different
normalisation. (The blueprint was checked locally on a skim of the complete first ttbar file: all 22 ttbar histograms
match the 1-file reference.)

In [ ]:
from agc.inputs import reference_file
from agc.validation import compared, validate

if DATA == "remote":
    ref = reference_file(N_FILES_PER_SAMPLE)
    errors = validate(OUTPUT, ref)
    print(f"Compared {len(compared(OUTPUT, ref))} histograms with {ref.name}")
    print("All good!" if not errors else "\n".join(f"{k}: {v}" for k, v in errors.items()))
else:
    print("Skipped: validation needs DATA = 'remote'.")

## 7. Clean up

Close the client and the cluster. On RS-DAT this cancels the worker jobs; the Jupyter job keeps running until you
stop it or its walltime ends.

In [ ]:
client.close()
cluster.close()

## References

- A. Held, E. Kauffman, O. Shadura, A. Wightman, *Physics analysis for the HL-LHC: Concepts and pipelines in practice with the Analysis Grand Challenge*, EPJ Web of Conferences 295, 06016 (2024), [doi:10.1051/epjconf/202429506016](https://doi.org/10.1051/epjconf/202429506016).
- V. E. Padulano, E. Guiraud, A. Falko, E. Gazzarrini, E. Garcia Garcia, D. Gosein, *First implementation and results of the Analysis Grand Challenge with a fully Pythonic RDataFrame*, EPJ Web of Conferences 295, 06011 (2024), [doi:10.1051/epjconf/202429506011](https://doi.org/10.1051/epjconf/202429506011).
- [iris-hep/analysis-grand-challenge](https://github.com/iris-hep/analysis-grand-challenge) (AGC: analysis definition, coffea implementation, file list and reference histograms; `analyses/cms-open-data-ttbar`).
- [root-project/analysis-grand-challenge](https://github.com/root-project/analysis-grand-challenge) (RDataFrame implementation).
- [CMS Open Data](https://opendata.cern.ch): the CMS 2015 simulated samples behind the AGC NanoAOD files (see the [AGC documentation](https://agc.readthedocs.io) for the datasets).
- Analysis code of this notebook: `../agc/rdf_analysis.py`; blueprint overview and Spider instructions: `../README.md`.